In [16]:
import z3
import random

def abs_z3(x):
    return z3.If(x >= 0, x, -x)

def solve_with_random_target():
    x = z3.Int('x')
    y = z3.Int('y')

    opt = z3.Optimize()
    
    # Constraints
    opt.add(x >= 0, x <= 100)
    opt.add(y >= 0, y <= 100)
    opt.add(x + y > 100)

    # Pick a random target
    target_x = random.randint(0, 100)
    target_y = random.randint(0, 100)

    # Minimize distance to target
    distance = abs_z3(x - target_x) + abs_z3(y - target_y)
    opt.minimize(distance)

    if opt.check() == z3.sat:
        m = opt.model()
        # FIX: Convert Z3 values to python ints using .as_long() 
        # so that the :3 formatting works correctly.
        sol_x = m[x].as_long()
        sol_y = m[y].as_long()
        
        print(f"Target: ({target_x:3}, {target_y:3}) | Solution: x={sol_x:3}, y={sol_y:3}")

print("Generating 10 diverse solutions using Target Points:")
for _ in range(10):
    solve_with_random_target()

Generating 10 diverse solutions using Target Points:
Target: (  3,  91) | Solution: x= 10, y= 91
Target: ( 81,  95) | Solution: x= 81, y= 95
Target: ( 54,  43) | Solution: x= 55, y= 46
Target: ( 33,   0) | Solution: x= 33, y= 68
Target: ( 93,  11) | Solution: x= 93, y= 11
Target: ( 77,  69) | Solution: x= 77, y= 69
Target: ( 28,  80) | Solution: x= 28, y= 80
Target: ( 79,   6) | Solution: x= 95, y=  6
Target: ( 77,  97) | Solution: x= 77, y= 97
Target: ( 86,  29) | Solution: x= 86, y= 29


In [9]:
import z3
import random
import time

def solve_diverse_all_vars(num_samples=100):
    start_time = time.time()
    # Use Optimize
    s = z3.Optimize()
    
    # 2 seconds is plenty for soft constraints
    s.set(timeout=2000) 
    
    ep, b, A, x, y, v = z3.Reals('ep b A x y v')
    dx, dy, xo, yo = z3.Reals('dx dy xo yo')
    all_vars = [ep, b, A, x, y, dx, dy, xo, yo, v]
    
    # Core Constraints (Hard)
    max_val = 100.0
    for var in [ep, b, A, x, y, dx, dy, xo, yo]:
        s.add(var > -max_val, var < max_val)
    s.add(A >= 0, b > 0, ep > 0, v == 0)
    s.add((x - xo)**2 + (y - yo)**2 > 0.001)
    s.add(dx**2 + dy**2 >= 0.9, dx**2 + dy**2 <= 1.1)

    results = []
    print(f"{'Sample':<8} | {'x':<8} | {'y':<8} | {'dx':<8} | {'dy':<8} | {'Time (s)':<10}")
    print("-" * 75)

    while len(results) < num_samples:
        iter_start = time.time()
        s.push()
        
        # Use add_soft instead of minimize. 
        # This is MUCH faster for generating many unique samples.
        for var in all_vars:
            if str(var) == 'v': continue
            target = random.uniform(-90, 90)
            s.add_soft(var == target, weight=1) 

        status = s.check()
        
        if status == z3.sat:
            m = s.model()
            def get_val(v):
                val = m[v]
                if val is None: return 0.0
                if z3.is_rational_value(val):
                    return float(val.as_fraction())
                return float(val.as_long())

            res = {str(var): get_val(var) for var in all_vars}
            results.append(res)
            
            idx = len(results)
            print(f"{idx:<8} | {res['x']:8.4f} | {res['y']:8.4f} | {res['dx']:8.4f} | {res['dy']:8.4f} | {time.time() - iter_start:8.2f}s")
            
            s.pop()
            # To ensure the 100 samples are actually different and not just 
            # 0.0000001 apart, we block a small area around the result.
            s.add(z3.Or(z3.Abs(x - m[x]) > 0.1, z3.Abs(y - m[y]) > 0.1))
        
        elif status == z3.unknown:
            s.pop()
            # If it times out, we just try again with new random targets
            continue
        else:
            s.pop()
            print("Search space exhausted.")
            break
        
    total_duration = time.time() - start_time
    print("-" * 75)
    print(f"Total Time: {total_duration:.2f}s for {len(results)} unique samples.")
    return results

data = solve_diverse_all_vars(100)

Sample   | x        | y        | dx       | dy       | Time (s)  
---------------------------------------------------------------------------
1        |  42.8307 |  34.4782 |  -0.2500 |   1.0000 |     0.01s
2        |  53.8846 |   5.1871 |   1.0000 |   0.0000 |     0.02s
3        |  39.5814 | -82.5993 |  -1.0000 |   0.0000 |     0.02s
4        |  59.3328 |  27.8246 |   0.0000 |   1.0000 |     0.03s
5        |  85.4783 | -82.3485 |  -1.0000 |   0.0000 |     0.03s
6        |  45.4052 |  60.6304 |   0.0000 |  -1.0312 |     0.05s
7        |  60.9590 | -18.0357 |   0.0000 |   1.0000 |     0.07s
8        |  78.5013 |  69.4828 |  -1.0000 |   0.0000 |     0.06s
9        | -60.7308 |  82.3068 |  -0.8750 |  -0.5000 |     0.06s
10       |  46.7506 | -59.4416 |  -1.0000 |   0.0000 |     0.04s
11       | -48.4322 |  13.3280 |   0.0000 |   1.0312 |     0.05s
12       | -57.0828 |   4.4506 |   1.0000 |   0.0000 |     0.13s
13       |  16.7736 |  49.8012 |  -1.0000 |   0.0000 |     0.14s
14       | -5

KeyboardInterrupt: 

In [10]:
import math
import random
import numpy as np
import re
from collections import Counter

# --- Mock Classes for Dependencies ---
# These classes simulate the configuration objects your environment would have.

class MockConstantDecl:
    """Mocks a constant declaration from a problem definition."""
    def __init__(self, name, value=None):
        self.name = name
        self.value = value

class MockDefinitionsConfig:
    """Mocks the definitions configuration, holding constants."""
    def __init__(self, constants):
        self.constants = constants

class MockPrecondition:
    """Mocks the precondition object, holding its formula."""
    def __init__(self, formula):
        self.formula = formula

class MockProblemConfig:
    """Mocks the problem configuration, holding the precondition."""
    def __init__(self, precondition_formula):
        self.precondition = MockPrecondition(precondition_formula)

# --- Test Environment Class ---
# This class hosts your _populate_cisb function and its necessary attributes.

class TestEnv:
    def __init__(self, cisb_size=10, cisb_populate_attempts=100, seed=42):
        self._cisb = []
        self._cisb_size = cisb_size
        self._cisb_populate_attempts = cisb_populate_attempts
        # Use a fixed seed for reproducibility of the test
        self.np_random = np.random.default_rng(seed=seed)

        # Mock essential attributes that _populate_cisb uses
        self._state_keys = ['x', 'y', 'v', 'dx', 'dy', 't', 'w', 'r', 'xo', 'yo']
        self._definitions_config = MockDefinitionsConfig(
            constants=[
                MockConstantDecl('ep', '1.0'),
                MockConstantDecl('b', '1.0'),
                MockConstantDecl('A', '0.5'),
                MockConstantDecl('W', '0.1'),
                MockConstantDecl('some_unfixed_const', None) # Example of a constant Z3 might solve for
            ]
        )
        # A simple, broad precondition that allows for diverse solutions
        self._problem_config = MockProblemConfig(precondition_formula="True") # Or "x > -100 && x < 100"

        # Initialize state and constants dictionaries
        self.state = {k: 0.0 for k in self._state_keys}
        self.constants = {c.name: float(c.value) if c.value and re.fullmatch(r'^-?\d+(\.\d+)?$', c.value) else 0.0 for c in self._definitions_config.constants}

        # Mock the Z3 solver function
        self._solve_initial_state_with_z3 = self._mock_z3_solver

    def _mock_z3_solver(self, formula):
        """
        Mocks the Z3 solver. It returns the current state and constant values
        as a solution, simulating Z3 finding a satisfying assignment based on
        the 'trial_values' passed into self.state/self.constants.
        """
        solution = {}
        # Combine state and constant values into a single solution dictionary
        for k in self._state_keys:
            solution[k] = self.state[k]
        for c_decl in self._definitions_config.constants:
            solution[c_decl.name] = self.constants[c_decl.name]

        # Simulate Z3 failing sometimes (e.g., precondition not met, timeout)
        # This ensures the 'attempts' counter logic is also exercised.
        if self.np_random.random() < 0.1: # 10% chance of Z3 "failing" to find a solution
            return None
        return solution

    # --- Your _populate_cisb function (copied as provided) ---
    def _populate_cisb(self):
        """
        Populates the Contrastive Initial State Buffer (CISB) with diverse initial states
        that satisfy the problem's precondition, using Z3 for verification.
        """
        print(f"Populating Contrastive Initial State Buffer (CISB) with {self._cisb_size} states...")
        current_cisb_count = 0
        attempts = 0

        # Define reasonable bounds for random sampling for state variables and constants.
        # These ranges should reflect the expected operational domain of the system.
        # Adjust these ranges based on the specific problem.
        sample_ranges = {
            'x': (-10.0, 10.0), 'y': (-10.0, 10.0), 'v': (0.0, 0.0), # v is fixed to 0 by precondition
            'dx': (-1.0, 1.0), 'dy': (-1.0, 1.0),
            'w': (-2.0, 2.0), # w is not fixed by precondition, can be anything
            'r': (0.1, 10.0), # r must be non-zero by precondition
            'xo': (-10.0, 10.0), 'yo': (-10.0, 10.0),
            # Constants that Z3 might solve for, or need to be within bounds
            'ep': (0.1, 5.0), # ep > 0 by bounds()
            'b': (0.1, 5.0),  # b > 0 by bounds()
            'A': (0.0, 5.0),  # A >= 0 by bounds()
            'W': (0.0, 2.0),   # W >= 0 (often implicit for magnitude)
            'some_unfixed_const': (-5.0, 5.0) # Range for our mock unfixed constant
        }

        # Collect all variable names that Z3 will be solving for (state vars + non-fixed constants)
        z3_target_vars = set(self._state_keys)
        for const_decl in self._definitions_config.constants:
            # Only add constants that don't have a fixed numeric value to z3_target_vars
            if not (const_decl.value is not None and re.fullmatch(r'^-?\d+(\.\d+)?$', const_decl.value)):
                z3_target_vars.add(const_decl.name)

        while current_cisb_count < self._cisb_size and attempts < self._cisb_populate_attempts:
            attempts += 1
            trial_values = {} # Stores sampled values for both state and constants

            # 1. Randomly sample values for relevant variables/constants within defined ranges
            for var_name in z3_target_vars:
                if var_name in sample_ranges:
                    trial_values[var_name] = self.np_random.uniform(*sample_ranges[var_name])
                else:
                    # Fallback for variables not in sample_ranges, use a wide default
                    trial_values[var_name] = self.np_random.uniform(-10.0, 10.0)

            # Heuristic for dx, dy to make `isWellformedDir` more likely to be satisfied by Z3.
            # Normalize them to approximately form a unit vector.
            dx_val = trial_values.get('dx', 0.0)
            dy_val = trial_values.get('dy', 0.0)
            magnitude = math.sqrt(dx_val**2 + dy_val**2)
            if magnitude > 1e-6:
                trial_values['dx'] = dx_val / magnitude
                trial_values['dy'] = dy_val / magnitude
            else:
                # If dx, dy are both near zero, set to a default unit vector
                trial_values['dx'] = 1.0
                trial_values['dy'] = 0.0

            # Ensure 'v' is 0.0 and 't' is 0.0 as per initialState and problem setup
            trial_values['v'] = 0.0
            trial_values['t'] = 0.0

            # 2. Temporarily apply sampled values to `self.state` and `self.constants`
            # This allows `_solve_initial_state_with_z3` to use these as hints/fixed values
            # and ensures `_evaluate_expression` (used internally by Z3 conversion) has access.
            original_state_copy = self.state.copy()
            original_constants_copy = self.constants.copy()

            for k, v in trial_values.items():
                if k in self.state:
                    self.state[k] = v
                elif k in self.constants:
                    self.constants[k] = v

            # 3. Use Z3 to find a precise satisfying assignment for the precondition
            z3_solution = self._solve_initial_state_with_z3(self._problem_config.precondition.formula)

            # 4. Restore original `self.state` and `self.constants`
            self.state.update(original_state_copy)
            self.constants.update(original_constants_copy)

            if z3_solution:
                # Store the Z3-verified solution in the buffer
                # The solution contains values for all state variables and relevant constants.
                self._cisb.append(z3_solution)
                current_cisb_count += 1
                if current_cisb_count % 100 == 0:
                    print(f"  ... {current_cisb_count}/{self._cisb_size} states populated.")

        if current_cisb_count < self._cisb_size:
            print(f"Warning: Could only populate {current_cisb_count} states for CISB after {attempts} attempts.")
        else:
            print(f"CISB population complete with {current_cisb_count} states.")
        if not self._cisb:
            print("CRITICAL ERROR: CISB is empty! This environment cannot function correctly without initial states.")
            print("Please check precondition and sampling ranges for satisfiability.")

# --- Test Execution ---
if __name__ == "__main__":
    # Initialize the test environment with a specific seed for reproducible results
    env = TestEnv(cisb_size=20, cisb_populate_attempts=200, seed=42)
    env._populate_cisb()

    print(f"\nGenerated {len(env._cisb)} states.")

    # Check for uniqueness of generated states
    unique_states = set()
    for state_dict in env._cisb:
        # To check for uniqueness, convert the dictionary to a hashable tuple.
        # Sort items to ensure consistent order, and format floats to a fixed precision
        # to avoid issues with floating-point comparisons.
        state_tuple = tuple(sorted((k, f"{v:.10f}") for k, v in state_dict.items()))
        unique_states.add(state_tuple)

    print(f"Number of unique states (after float precision adjustment): {len(unique_states)}")

    if len(unique_states) == len(env._cisb):
        print("Result: All generated states are unique!")
    else:
        num_duplicates = len(env._cisb) - len(unique_states)
        print(f"Result: Found {num_duplicates} duplicate states.")
        # Optional: Print duplicate states for debugging
        # all_states_tuples = [tuple(sorted((k, f"{v:.10f}") for k, v in s.items())) for s in env._cisb]
        # counts = Counter(all_states_tuples)
        # for state, count in counts.items():
        #     if count > 1:
        #         print(f"  Duplicate state (appears {count} times): {dict(state)}")

Populating Contrastive Initial State Buffer (CISB) with 20 states...
CISB population complete with 20 states.

Generated 20 states.
Number of unique states (after float precision adjustment): 20
Result: All generated states are unique!
